In [ ]:
import cv2
import numpy as np
import pandas as pd

video_path = "road-traffic.mp4"  
pixel_to_meter = 0.05  # 1 pixel = 5 cm
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
dt = 1 / fps
print(f"FPS: {fps}, dt: {dt:.4f} sec/frame")

# --- Background subtraction for motion detection ---
fgbg = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=50)

# --- Tracking data ---
prev_centers = {}
velocities = {}  # {vehicle_id: [velocities]}
frame_index = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Resize for faster processing 
    frame = cv2.resize(frame, (960, 540))

    # Apply background subtraction
    fgmask = fgbg.apply(frame)

    # Morphology to reduce noise
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    fgmask = cv2.morphologyEx(fgmask, cv2.MORPH_OPEN, kernel)
    fgmask = cv2.morphologyEx(fgmask, cv2.MORPH_DILATE, kernel, iterations=2)

    # Find contours of moving objects
    contours, _ = cv2.findContours(fgmask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    current_centers = {}

    for i, c in enumerate(contours):
        area = cv2.contourArea(c)
        if area < 500:  # filter small objects
            continue

        x, y, w, h = cv2.boundingRect(c)
        center = (int(x + w/2), int(y + h/2))
        current_centers[i] = center

        # Draw bounding box
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.circle(frame, center, 5, (0, 0, 255), -1)

        # Compute velocity if previous frame exists
        if i in prev_centers:
            dx = center[0] - prev_centers[i][0]
            dy = center[1] - prev_centers[i][1]
            displacement_px = np.sqrt(dx**2 + dy**2)
            displacement_m = displacement_px * pixel_to_meter
            velocity_mps = displacement_m / dt
            velocity_kmh = velocity_mps * 3.6

            # Save velocity for this vehicle
            velocities.setdefault(i, []).append(velocity_kmh)

            # Draw speed on frame
            cv2.putText(frame, f"{velocity_kmh:.1f} km/h", (x, y-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

    prev_centers = current_centers
    frame_index += 1

    cv2.imshow("Vehicle Speed Detection", frame)

    if cv2.waitKey(30) & 0xFF == 27:  # ESC to exit
        break

cap.release()
cv2.destroyAllWindows()

# --- Compute average velocity per vehicle ---
avg_velocities = {vid: np.mean(vlist) for vid, vlist in velocities.items()}
print("\nAverage velocity of each vehicle (km/h):")
for vid, avg in avg_velocities.items():
    print(f"Vehicle {vid}: {avg:.2f} km/h")

# --- Save to CSV ---
df = pd.DataFrame(list(avg_velocities.items()), columns=["VehicleID", "Average Velocity (km/h)"])
df.to_csv("average_vehicle_velocity.csv", index=False)
print("Average velocities saved to average_vehicle_velocity.csv")




FPS: 25.0, dt: 0.0400 sec/frame

Average velocity of each vehicle (km/h):
Vehicle 0: 18.92 km/h
Vehicle 62: 272.83 km/h
Vehicle 66: 314.62 km/h
Vehicle 71: 102.70 km/h
Vehicle 6: 106.26 km/h
Vehicle 67: 310.10 km/h
Vehicle 64: 124.12 km/h
Vehicle 65: 192.79 km/h
Vehicle 69: 142.19 km/h
Vehicle 23: 208.95 km/h
Vehicle 68: 290.78 km/h
Vehicle 60: 204.42 km/h
Vehicle 58: 185.84 km/h
Vehicle 61: 226.48 km/h
Vehicle 47: 166.08 km/h
Vehicle 51: 246.39 km/h
Vehicle 46: 228.27 km/h
Vehicle 48: 215.82 km/h
Vehicle 21: 207.80 km/h
Vehicle 52: 255.14 km/h
Vehicle 4: 161.79 km/h
Vehicle 3: 102.25 km/h
Vehicle 5: 124.58 km/h
Vehicle 1: 48.95 km/h
Vehicle 2: 66.33 km/h
Vehicle 7: 136.94 km/h
Vehicle 31: 229.03 km/h
Vehicle 18: 31.26 km/h
Vehicle 54: 212.01 km/h
Vehicle 55: 167.62 km/h
Vehicle 49: 226.91 km/h
Vehicle 8: 56.63 km/h
Vehicle 10: 162.47 km/h
Vehicle 50: 207.06 km/h
Vehicle 15: 53.27 km/h
Vehicle 28: 140.24 km/h
Vehicle 41: 173.85 km/h
Vehicle 43: 192.70 km/h
Vehicle 12: 26.34 km/h
Vehicl